# Capstone pipeline — Riccio / Kaggle + BiLSTM (Google Colab)

End-to-end path for **[Real-time exercise recognition (Riccio)](https://www.kaggle.com/datasets/riccardoriccio/real-time-exercise-recognition-dataset)** → MediaPipe angles/keypoints → **BiLSTM** (and optional **ST-GCN**).

| Step | What |
|------|------|
| Data | Kaggle dataset via **kagglehub** (`--download`) or upload prebuilt NPZs |
| Pose | **MediaPipe** in `riccio_kaggle_video_pipeline.py` |
| Features | Preprocessing + **8 joint angles** in `*_biomechanics.npz`; optional `*_keypoints.npz` |
| Train | `train_exercise_bilstm.py --kaggle-angles-dir` (per-video split when `video_id` in labels) |
| Optional | `train_exercise_stgcn.py --kaggle-keypoints-dir` (needs keypoints in NPZ folder) |

**Setup:** Runtime → **GPU**. **Secrets:** `KAGGLE_USERNAME` + `KAGGLE_KEY` ([API token](https://www.kaggle.com/settings)) for `kagglehub` download. Optional: `GITHUB_TOKEN` to clone a private repo.

**Disk:** Colab VM disk is small. Use **Google Drive** for `RESULTS` and the Riccio pipeline output, or upload NPZs you built locally.

> **Also:** see [`colab_gpu_training.ipynb`](colab_gpu_training.ipynb) (minimal BiLSTM-only) and [`colab_egoexo_capstone_pipeline.ipynb`](colab_egoexo_capstone_pipeline.ipynb) (EgoExo).

In [ ]:
!nvidia-smi -L || true
import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

## 1) Clone repo + install

Change `REPO_URL` to your fork, or mount Drive and `cd` into the project.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USER/Finess-coach-capstone-1.git"
BRANCH = "main"
WORK = Path("/content/fitness-coach")

if not WORK.is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(WORK)], check=True)
os.chdir(WORK)
print("cwd:", os.getcwd())

In [ ]:
%pip install -q -r requirements.txt
%pip install -q kagglehub
%pip install -q -e .

import torch
print("cuda:", torch.cuda.is_available())

## 2) Kaggle credentials (for `kagglehub` download)

Colab **Secrets:** `KAGGLE_USERNAME`, `KAGGLE_KEY` (create at [Kaggle → Account → API](https://www.kaggle.com/settings)).

The hub writes `~/.kaggle/kaggle.json` from the environment if needed.

In [ ]:
import json
import os
from pathlib import Path

try:
    from google.colab import userdata
    u = userdata.get("KAGGLE_USERNAME")
    k = userdata.get("KAGGLE_KEY")
except Exception as e:
    print("Colab Secrets missing?", e)
    u = k = None

if u and k:
    p = Path.home() / ".kaggle"
    p.mkdir(parents=True, exist_ok=True)
    (p / "kaggle.json").write_text(json.dumps({"username": u, "key": k}))
    os.chmod(p / "kaggle.json", 0o600)
    print("Wrote ~/.kaggle/kaggle.json")
else:
    print("Set KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets, or skip download and upload NPZs.")

## 3) Output folders (recommend Google Drive)

Avoid filling Colab ephemeral disk.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
RESULTS = Path("/content/drive/MyDrive/fitness_coach_kaggle_results")
RESULTS.mkdir(parents=True, exist_ok=True)
KAGGLE_DIR = RESULTS / "riccio_realtime_exercise_recognition"
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
STEM = "riccio_realtime_exercise_recognition"
print("KAGGLE_DIR:", KAGGLE_DIR)
print("RESULTS:", RESULTS)

## 4) Build Riccio NPZs (MediaPipe + preprocessing)

**First run** downloads the dataset via **kagglehub** (slow, large). **GPU does not speed up** MediaPipe here (CPU).

- Omit `--skip-keypoints` if you plan **ST-GCN** later (writes large `*_keypoints.npz`).
- Use `--skip-keypoints` for a faster, BiLSTM-only (angles) run.

If you already have NPZs (e.g. from your laptop), **skip this cell** and set `KAGGLE_DIR` to that folder.

In [ ]:
import subprocess
import sys

# Long-running: download + all videos. Reduce scope with pipeline flags if your repo supports them.
cmd = [
    sys.executable,
    "riccio_kaggle_video_pipeline.py",
    "--download",
    "--output-dir", str(KAGGLE_DIR),
    "--output-stem", STEM,
    "--workers", "0",
    "--skip-keypoints",  # remove this line for ST-GCN keypoints
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=str(WORK), check=True)

## 5) Train BiLSTM (Kaggle NPZ mode)

- **`--preset riccio`**: Table-style hyperparameters.
- **`--eval-test`**: test accuracy + quality RMSE + **macro F1** + per-class F1 (see `test_classification_metrics.json`).
- **Dual heads (class + quality):** use the **first** block (omit `--classification-only`). Riccio labels often use a **default quality** unless you extended the pipeline.
- **Classification-only** (matches `pipeline/step_05_train_bilstm_riccio.sh`): use the **second** block.

In [ ]:
import subprocess
import sys

OUT_BILSTM = RESULTS / "exercise_bilstm_kaggle"

assert (KAGGLE_DIR / f"{STEM}_biomechanics.npz").is_file(), "Run pipeline or upload NPZs"

cmd = [
    sys.executable, "train_exercise_bilstm.py",
    "--preset", "riccio",
    "--standardize",
    "--eval-test",
    "--architecture", "cnn_attn",
    "--epochs", "30",
    "--kaggle-angles-dir", str(KAGGLE_DIR),
    "--kaggle-stem", STEM,
    "--output-dir", str(OUT_BILSTM),
    "--kaggle-seed", "42",
    "--balanced-class-weights",
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=str(WORK), check=True)

In [ ]:
# Optional: classification-only (faster ablation; no regression head loss)
# import subprocess, sys
# subprocess.run([
#     sys.executable, "train_exercise_bilstm.py",
#     "--preset", "riccio", "--standardize", "--eval-test",
#     "--architecture", "cnn_attn", "--classification-only",
#     "--epochs", "30",
#     "--kaggle-angles-dir", str(KAGGLE_DIR), "--kaggle-stem", STEM,
#     "--output-dir", str(RESULTS / "exercise_bilstm_kaggle_cls_only"),
#     "--kaggle-seed", "42",
# ], cwd=str(WORK), check=True)

In [ ]:
import json
from pathlib import Path
OUT_BILSTM = RESULTS / "exercise_bilstm_kaggle"
p = OUT_BILSTM / "test_classification_metrics.json"
if p.is_file():
    print(json.dumps(json.loads(p.read_text()), indent=2))
else:
    print("No test metrics yet — train with --eval-test")

## 6) Optional — ST-GCN (needs `*_keypoints.npz`)

Re-run **section 4** without `--skip-keypoints`, then:

In [ ]:
import subprocess
import sys

kp = KAGGLE_DIR / f"{STEM}_keypoints.npz"
if not kp.is_file():
    print("Skip: no keypoints NPZ — rebuild pipeline without --skip-keypoints")
else:
    subprocess.run([
        sys.executable, "train_exercise_stgcn.py",
        "--kaggle-keypoints-dir", str(KAGGLE_DIR),
        "--kaggle-stem", STEM,
        "--standardize",
        "--epochs", "50",
        "--output-dir", str(RESULTS / "stgcn_kaggle"),
        "--kaggle-seed", "42",
    ], cwd=str(WORK), check=True)

## 7) Ablation reminder (angles vs coordinates)

For **index-CSV** workflows you can compare `--feature-mode angles` vs `coords`. In **pure Kaggle NPZ** mode, BiLSTM uses **angles only**; coordinate ablations need an index + `keypoints_dir` or a second training path — see `pipeline/README.md`.

## 8) Download artifacts

```python
from google.colab import files
files.download(str(OUT_BILSTM / "exercise_bilstm_best.pt"))
```

Or keep everything under `RESULTS` on Drive.

---
## Running on **kaggle.com** (native Kaggle Notebook)

1. Add the **Real-time exercise recognition** dataset as input.
2. Add this repo (Code → Add dataset / upload wheel / clone).
3. Set `EXERCISE_RECOGNITION_ROOT` or `KAGGLE_DIR` to `/kaggle/input/...`.
4. Run the same CLI commands in terminal cells; **no** `drive.mount`.

Secrets: use Kaggle’s **Add-ons → Secrets** for API keys if you pull extra data via hub.